# FalkeAI Own-Voice Fine-Tuning Studio ($0 GPU on Kaggle or Google Colab)

This notebook takes the **`falke_voice_dataset.zip`** you recorded inside **FalkeAI -> Voice -> Clone Voice**, fine-tunes a lightweight single-speaker **Piper / VITS** neural voice model on a free T4 GPU, and exports **`falcon_custom_voice.zip`** (`model.onnx` + `tokens.txt` + `lexicon.txt`) ready to import straight into your Infinix Hot 10T for **100% offline, $0 speech in your own voice**.

### Before Running
1. Enable a free **T4 GPU** (**Runtime -> Change runtime type -> T4 GPU** in Colab, or **Accelerator -> GPU T4 x2** in Kaggle).
2. Upload `falke_voice_dataset.zip` to `/content/falke_voice_dataset.zip` (or Kaggle Input).

In [ ]:
# 1. Verify GPU & Install Piper Training + sherpa-onnx Export Tools
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq espeak-ng build-essential cmake ninja-build rsync zip
!pip install -q cython>=0.29 numpy==1.26.4 torch torchaudio pytorch-lightning==1.9.5 librosa onnx onnxruntime phonemizer piper-phonemize

In [ ]:
# 2. Clone Official Piper Training Pipeline & Unpack Your FalkeAI Voice Dataset
import os, zipfile, shutil

if not os.path.exists('/content/piper'):
    !git clone --depth 1 https://github.com/rhasspy/piper.git /content/piper
!pip install -q -e /content/piper/src/python

DATASET_ZIP = '/content/falke_voice_dataset.zip'
assert os.path.exists(DATASET_ZIP), 'Please upload falke_voice_dataset.zip to /content/ first!'

RAW_DIR = '/content/falcon_dataset'
shutil.rmtree(RAW_DIR, ignore_errors=True)
os.makedirs(RAW_DIR, exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
    z.extractall(RAW_DIR)

wav_count = len([f for f in os.listdir(os.path.join(RAW_DIR, 'wavs')) if f.endswith('.wav')])
print(f'Unpacked {wav_count} studio WAV clips ready for preprocessing!')

In [ ]:
# 3. Preprocess Dataset (22,050 Hz Single-Speaker English + Download Pretrained Base Checkpoint)
WORK_DIR = '/content/falcon_training_work'
shutil.rmtree(WORK_DIR, ignore_errors=True)

!python3 -m piper_train.preprocess \
  --language en-us \
  --input-dir /content/falcon_dataset \
  --output-dir /content/falcon_training_work \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050

# Download pretrained Lessac Medium checkpoint so fine-tuning converges cleanly in ~30-45 mins!
BASE_CKPT = '/content/base_lessac_medium.ckpt'
if not os.path.exists(BASE_CKPT):
    !curl -fL -o /content/base_lessac_medium.ckpt \
      'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'

In [ ]:
# 4. Fine-Tune on Your Voice (~400 additional epochs over base checkpoint)
# Note: Base checkpoint is epoch 2164; setting --max_epochs 2564 runs 400 fine-tuning epochs (~35 mins on T4).
!python3 -m piper_train \
  --dataset-dir /content/falcon_training_work \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.05 \
  --num-test-examples 2 \
  --max_epochs 2564 \
  --resume_from_checkpoint /content/base_lessac_medium.ckpt \
  --checkpoint-epochs 50 \
  --precision 16

In [ ]:
# 5. Export to sherpa-onnx v2.0+ Compatible Bundle (model.onnx + tokens.txt + lexicon.txt) -> falcon_custom_voice.zip
import glob, json, subprocess

ckpts = sorted(glob.glob('/content/falcon_training_work/lightning_logs/**/checkpoints/*.ckpt', recursive=True), key=os.path.getmtime)
assert ckpts, 'No trained checkpoint found yet!'
latest_ckpt = ckpts[-1]
print('Exporting checkpoint:', latest_ckpt)

EXPORT_DIR = '/content/falcon_export'
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

!python3 -m piper_train.export_onnx "{latest_ckpt}" /content/falcon_export/model.onnx

# Convert Piper config.json phoneme_id_map -> sherpa-onnx tokens.txt
with open('/content/falcon_training_work/config.json', 'r', encoding='utf-8') as f:
    cfg = json.load(f)

valid_tokens = set(cfg['phoneme_id_map'].keys())
with open('/content/falcon_export/tokens.txt', 'w', encoding='utf-8') as out:
    for sym, ids in cfg['phoneme_id_map'].items():
        for idx in ids:
            out.write(f'{sym} {idx}\n')

# Add sherpa-onnx ONNX metadata so OfflineTts VITS loads cleanly on Android
import onnx
m = onnx.load('/content/falcon_export/model.onnx')
meta = {
    'model_type': 'vits',
    'comment': 'piper',
    'language': 'English',
    'voice': 'en-us',
    'has_espeak': '1',
    'n_speakers': '1',
    'sample_rate': str(cfg['audio']['sample_rate']),
    'noise_scale': '0.667',
    'noise_scale_w': '0.8',
    'length_scale': '1.0'
}
for k, v in meta.items():
    p = m.metadata_props.add()
    p.key = k
    p.value = v
onnx.save(m, '/content/falcon_export/model.onnx')

# Download official sherpa-onnx v2.0+ English IPA lexicon & append custom Fall-K / Nigerian entries
!curl -fL -o /content/falcon_export/lexicon.txt \
  'https://github.com/k2-fsa/sherpa-onnx/releases/download/tts-models/lexicon-en-us.txt'

def ipa_phones(spoken_phrase):
    raw = subprocess.check_output(['espeak-ng', '-v', 'en-us', '-q', '--ipa', spoken_phrase], text=True).strip()
    chars = [c for c in raw if c in valid_tokens and not c.isspace()]
    return ' '.join(chars)

custom_words = {
    'falke': 'Fall Kay',
    'fall-kay': 'Fall Kay',
    'fall-k': 'Fall Kay',
    'falkeai': 'Fall Kay A I',
    'aurikrex': 'Au rik rex',
    'naira': 'Nigh rah',
    'wahala': 'wah hah lah'
}
with open('/content/falcon_export/lexicon.txt', 'a', encoding='utf-8') as lex:
    lex.write('\n# FalkeAI custom pronunciations\n')
    for word, respelling in custom_words.items():
        ph = ipa_phones(respelling)
        if ph:
            lex.write(f'{word} || {ph}\n')

OUT_ZIP = '/content/falcon_custom_voice.zip'
shutil.make_archive('/content/falcon_custom_voice', 'zip', EXPORT_DIR)
print(f'SUCCESS! Download {OUT_ZIP} ({os.path.getsize(OUT_ZIP) // (1024*1024)} MB) and tap Import Voice ZIP inside FalkeAI!')